In [1]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Setup and authentication complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: Connection error trying to communicate with service.


In [2]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUBKEY")
    os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN
    print("✅ GITHUB_TOKEN API key setup complete.")

except Exception as e:
     print(
        f"Authentication Error: Please make sure you have added 'GITHUBKEY' to your Kaggle secrets. Details: {e}"
    )

Authentication Error: Please make sure you have added 'GITHUBKEY' to your Kaggle secrets. Details: Connection error trying to communicate with service.


In [3]:
from google.genai import types

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search, AgentTool, FunctionTool, ToolContext
from google.adk.code_executors import BuiltInCodeExecutor
from google.adk.sessions import DatabaseSessionService
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


In [4]:
retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

In [5]:
pip install requests

Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import requests
from typing import List, Dict

# Define the GitHub API URL structure
GITHUB_API_URL = "https://api.github.com"

def find_missing_commits(repo_name: str, base_branch: str, compare_branch: str) -> str:
    """
    Compares two branches and returns a formatted list of commits in 
    the compare_branch that are missing from the base_branch.
    
    Args:
        repo_name: The repository name in 'owner/repo' format (e.g., 'google/adk-python').
        base_branch: The name of the branch to compare against (e.g., 'main').
        compare_branch: The name of the branch being checked for new commits (e.g., 'feature-branch').
        
    Returns:
        A string summarizing and listing the missing commits, or an error message.
    """
    
    try:
        # 1. Get Authentication Token
        # Use a Personal Access Token (PAT) for authenticated access
        github_token = os.getenv("GITHUB_TOKEN")
        if not github_token:
           return "ERROR: GITHUB_TOKEN environment variable is not set."

        # 2. Construct the API URL
        # The syntax for comparison is {base}...{head}
        comparison_endpoint = f"/repos/{repo_name}/compare/{base_branch}...{compare_branch}"
        api_url = GITHUB_API_URL + comparison_endpoint

        print(f" API URL is '{api_url}'")
        # 3. Define Headers
        headers = {
            "Authorization": f"token {github_token}",
            "Accept": "application/vnd.github.v3+json"
        }
        
        # 4. Make the API Call
        response = requests.get(api_url, headers=headers)
        response.raise_for_status() # Raises an HTTPError for bad responses (4xx or 5xx)
        data = response.json()

        # 5. Process the Result
        # The 'commits' list contains all commits from 'base' to 'head'. 
        # The 'ahead_by' field tells you how many commits 'compare_branch' is ahead.
        ahead_by = data.get("ahead_by", 0)
        
        if ahead_by == 0:
            return f"The branch '{compare_branch}' is not ahead of '{base_branch}'. No missing commits found."

       
        missing_commits: List[Dict] = data.get("commits", [])
        
        # Format the output for the LLM Agent
        commit_summaries = []
        for commit_data in missing_commits:
            sha = commit_data.get("sha", "")[:7] # Shorten SHA
            message = commit_data.get("commit", {}).get("message", "").split('\n')[0] # First line of message
            author = commit_data.get("commit", {}).get("author", {}).get("name", "Unknown")
            commit_summaries.append(f"- **{sha}**: {message} (Author: {author})")
        
        result_message = [
            f"Successfully compared **{repo_name}** branches.",
            f"Branch **'{compare_branch}'** is **{ahead_by}** commits ahead of **'{base_branch}'**.",
            "---",
            "Missing Commits (in compare_branch, not in base_branch):",
            *commit_summaries
        ]
        
        return "\n".join(result_message)

    except requests.exceptions.HTTPError as e:
        return f"ERROR: GitHub API call failed for repo '{repo_name}'. Status Code: {e.response.status_code}. Details: {e.response.text}"
    except Exception as e:
        return f"An unexpected error occurred: {e}"

In [7]:
compare_commits_tool = FunctionTool(func=find_missing_commits)

In [8]:
agent_instruction = """
You are a Git Branch Comparison Agent. Your primary function is to use the 
`find_missing_commits` tool to compare two branches in a specified GitHub 
repository. You must be provided with the repository name (e.g., 'owner/repo'), 
the base branch, and the compare branch. Your final output must clearly list the 
commits found in the compare branch that are missing from the base branch.
"""

# Create the LLM Agent
github_comp_agent = LlmAgent(
    name="github_comp_agent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction=agent_instruction,
    tools=[compare_commits_tool]
)

print("✅ github_comp_agent created.")

✅ github_comp_agent created.


In [9]:
# Stop and restart session when attempting to retry with changes , especially if sessionID DBsessionmanagement is modified

REPO_URL = "keerthimudireddy/Repository1"

BASE_BRANCH = "feature1"
TARGET_BRANCH = "main"
my_session = "DBsessionmanagement"

#response = InMemoryRunner(agent=github_comp_agent)
#_ = await response.run_debug(
#    f"Can you give me missing GitHub missing commits? repo_name is '{REPO_URL}', base branch to compare with is '{BASE_BRANCH}' and compare branch to compare is '{TARGET_BRANCH}' "
#)

db_url = "sqlite:///my_agent_data.db"  # Local SQLite file
session_service = DatabaseSessionService(db_url=db_url)

#runner = Runner(agent=github_comp_agent, app_name="default", session_service=session_service)
## commented this code below, because writing another agent which gives detailed summary.
#this agent execution can be uncommented if we only want to see the missing commits.
#await runner.run_debug(
 #   f"Can you give me missing GitHub missing commits, Is it safe to merge code? repo_name is '{REPO_URL}', base branch to compare with is '{BASE_BRANCH}' and compare branch to compare is '{TARGET_BRANCH}' ",
 #   session_id=my_session
#)

In [10]:
import os
import requests
from typing import List, Union

# Define the GitHub API URL structure
GITHUB_API_URL = "https://api.github.com"

def get_files_modified_by_commit(repo_name: str, commit_sha: str) -> Union[List[str], str]:
    """
    Retrieves the list of file paths that were modified in a specific Git commit SHA.
    
    This is essential for the agent to know which files to compare for content analysis.
    
    Args:
        repo_name: The repository name in 'owner/repo' format (e.g., 'google/adk-python').
        commit_sha: The full or shortened SHA of the commit to inspect.
        
    Returns:
        A list of modified file paths (strings), or an error message string.
    """
    
    try:
        # 1. Get Authentication Token
        github_token = os.getenv("GITHUB_TOKEN")
        if not github_token:
            return "ERROR: GITHUB_TOKEN environment variable is not set."

        # 2. Construct the API URL for a single commit
        commit_endpoint = f"/repos/{repo_name}/commits/{commit_sha}"
        api_url = GITHUB_API_URL + commit_endpoint

        # 3. Define Headers
        headers = {
            "Authorization": f"token {github_token}",
            "Accept": "application/vnd.github.v3+json"
        }
        
        # 4. Make the API Call
        response = requests.get(api_url, headers=headers)
        response.raise_for_status()
        data = response.json()

        # 5. Extract File Paths
        # The 'files' key in the commit response contains details on all modified files
        modified_files = []
        for file_data in data.get("files", []):
            # 'filename' contains the full path (e.g., 'src/component.js')
            modified_files.append(file_data.get("filename"))
        
        if not modified_files:
             return f"No file paths found for commit {commit_sha}. It might be a merge commit with no direct file changes."

        # 6. Return structured data (list of paths) for the LLM to use
        # Returning a list of paths makes it easy for the LLM to iterate and call the next tool.
        return modified_files

    except requests.exceptions.HTTPError as e:
        return f"ERROR: GitHub API call failed for commit {commit_sha}. Status Code: {e.response.status_code}. Details: {e.response.text}"
    except Exception as e:
        return f"An unexpected error occurred: {e}"

In [11]:
commit_modifiedfiles_tool = FunctionTool(func=get_files_modified_by_commit)

In [12]:
import os
import requests
from typing import Dict, List, Any
import hashlib

# Define the GitHub API URL structure
GITHUB_API_URL = "https://api.github.com"
# ... (Your existing find_missing_commits function goes here) ...

def get_file_content_at_commit(repo_name: str, path: str, ref: str) -> Dict[str, Any]:
    """
    Fetches the content, type, and SHA of a specific file at a given reference (branch/commit).
    
    Args:
        repo_name: The repository name in 'owner/repo' format.
        path: The path to the file in the repository (e.g., 'src/main.py').
        ref: The branch name or commit SHA to check (e.g., 'main' or 'a1b2c3d').
        
    Returns:
        A dictionary containing file details, or an error message.
    """
    
    try:
        github_token = os.getenv("GITHUB_TOKEN")
        if not github_token:
            return {"error": "GITHUB_TOKEN environment variable is not set."}

        # 1. Construct the API URL for contents
        content_endpoint = f"/repos/{repo_name}/contents/{path}"
        api_url = GITHUB_API_URL + content_endpoint
        
        # 2. Define Headers and Parameters
        headers = {
            "Authorization": f"token {github_token}",
            "Accept": "application/vnd.github.v3.raw"  # Request raw content directly
        }
        params = {
            "ref": ref  # Specify the branch or SHA
        }

        # 3. Make the API Call
        response = requests.get(api_url, headers=headers, params=params)
        
        if response.status_code == 404:
            # File not found at this reference is a key piece of information
            return {"error": f"File not found at path '{path}' in ref '{ref}'."}
            
        response.raise_for_status()
        
        # 4. Return the raw content and basic metadata
        # Since we used 'application/vnd.github.v3.raw', the response.text is the content itself.
        return {
            "content_sha256": hashlib.sha256(response.text.encode('utf-8')).hexdigest(), # Use SHA256 hash for comparison
            "content_length": len(response.text),
            "raw_content": response.text[:200] + "..." # Include a snippet for debugging/display
        }

    except requests.exceptions.HTTPError as e:
        return {"error": f"API call failed. Status Code: {e.response.status_code}"}
    except Exception as e:
        return {"error": f"An unexpected error occurred: {e}"}

In [13]:
compare_eachcommit_tool = FunctionTool(func=get_file_content_at_commit)

In [14]:
agent_instruction = """
You are a sophisticated Git Merge Safety Analyst. Your goal is to determine the safety of merging a branch based on file-by-file content comparison, not just commit history.

1. **First, use the `find_missing_commits` tool.** You must get the list of commits (SHAs) that are in the compare branch but missing from the base branch. This tool must return a list of commit objects.
2. **If missing commits are found, proceed to CONTENT ANALYSIS for each commit:**
    a. **Identify Files:** Get the list of files modified by the missing commit (You will need an intermediary tool/API call for this, as `find_missing_commits` only returns the commit list).
    b. **Fetch Content:** For every file in the missing commit, use the `get_file_content_at_commit` tool to fetch its content's SHA256 hash at both the **missing commit SHA** and the **base branch HEAD**.
3. **Determine the Verdict (CRITICAL STEP):**
    * **VERDICT A (SAFE MERGE):** If the file exists in the base branch AND the content (SHA256 hash) is IDENTICAL, conclude that the code is identical, and a standard merge is safe.
    * **VERDICT B (UNSAFE MERGE):** If the file DOES NOT EXIST in the base branch (404 error from content tool), conclude that the merge is **NOT advised** due to missing dependencies/files.
    * **VERDICT C (STANDARD MERGE):** If the file exists but the content is DIFFERENT, conclude that substantive changes exist, and a standard merge is required.
4. **Final Output:** Provide a clear, actionable summary for the user based on the most severe verdict found (e.g., if one commit is VERDICT B, the overall merge is unsafe).
"""

# Create the LLM Agent
github_commit_comp_agent = LlmAgent(
    name="github_commit_comp_agent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction=agent_instruction,
    tools=[compare_commits_tool,compare_eachcommit_tool,commit_modifiedfiles_tool]
)

print("✅ github_comp_each_commit_agent created.")

✅ github_comp_each_commit_agent created.


In [15]:
# Stop and restart session when attempting to retry with changes , especially if sessionID DBsessionmanagement is modified

REPO_URL = "keerthimudireddy/Repository1"

BASE_BRANCH = "feature1"
TARGET_BRANCH = "main"
my_session = "DBsessionmanagement"

#response = InMemoryRunner(agent=github_comp_agent)
#_ = await response.run_debug(
#    f"Can you give me missing GitHub missing commits? repo_name is '{REPO_URL}', base branch to compare with is '{BASE_BRANCH}' and compare branch to compare is '{TARGET_BRANCH}' "
#)

db_url = "sqlite:///my_agent_data.db"  # Local SQLite file
session_service = DatabaseSessionService(db_url=db_url)

runner = Runner(agent=github_commit_comp_agent, app_name="default", session_service=session_service)
await runner.run_debug(
    f"Can you give me missing GitHub commits, and crucially, "
    f"is it safe to merge the code based on **content analysis**? "
    f"repo_name is '{REPO_URL}', base branch to compare with is '{BASE_BRANCH}' "
    f"and compare branch to compare is '{TARGET_BRANCH}'",
    session_id=my_session
)


 ### Created new session: DBsessionmanagement

User > Can you give me missing GitHub commits, and crucially, is it safe to merge the code based on **content analysis**? repo_name is 'keerthimudireddy/Repository1', base branch to compare with is 'feature1' and compare branch to compare is 'main'


ValueError: Missing key inputs argument! To use the Google AI API, provide (`api_key`) arguments. To use the Google Cloud API, provide (`vertexai`, `project` & `location`) arguments.

In [ ]:
from IPython.core.display import display, HTML
from jupyter_server.serverapp import list_running_servers


# Gets the proxied URL in the Kaggle Notebooks environment
def get_adk_proxy_url():
    PROXY_HOST = "https://kkb-production.jupyter-proxy.kaggle.net"
    ADK_PORT = "8000"

    servers = list(list_running_servers())
    if not servers:
        raise Exception("No running Jupyter servers found.")

    baseURL = servers[0]["base_url"]

    try:
        path_parts = baseURL.split("/")
        kernel = path_parts[2]
        token = path_parts[3]
    except IndexError:
        raise Exception(f"Could not parse kernel/token from base URL: {baseURL}")

    url_prefix = f"/k/{kernel}/{token}/proxy/proxy/{ADK_PORT}"
    url = f"{PROXY_HOST}{url_prefix}"

    styled_html = f"""
    <div style="padding: 15px; border: 2px solid #f0ad4e; border-radius: 8px; background-color: #fef9f0; margin: 20px 0;">
        <div style="font-family: sans-serif; margin-bottom: 12px; color: #333; font-size: 1.1em;">
            <strong>⚠️ IMPORTANT: Action Required</strong>
        </div>
        <div style="font-family: sans-serif; margin-bottom: 15px; color: #333; line-height: 1.5;">
            The ADK web UI is <strong>not running yet</strong>. You must start it in the next cell.
            <ol style="margin-top: 10px; padding-left: 20px;">
                <li style="margin-bottom: 5px;"><strong>Run the next cell</strong> (the one with <code>!adk web ...</code>) to start the ADK web UI.</li>
                <li style="margin-bottom: 5px;">Wait for that cell to show it is "Running" (it will not "complete").</li>
                <li>Once it's running, <strong>return to this button</strong> and click it to open the UI.</li>
            </ol>
            <em style="font-size: 0.9em; color: #555;">(If you click the button before running the next cell, you will get a 500 error.)</em>
        </div>
        <a href='{url}' target='_blank' style="
            display: inline-block; background-color: #1a73e8; color: white; padding: 10px 20px;
            text-decoration: none; border-radius: 25px; font-family: sans-serif; font-weight: 500;
            box-shadow: 0 2px 5px rgba(0,0,0,0.2); transition: all 0.2s ease;">
            Open ADK Web UI (after running cell below) ↗
        </a>
    </div>
    """

    display(HTML(styled_html))

    return url_prefix


print("✅ Helper functions defined.")

In [ ]:
import logging
import os

# Clean up any previous logs
for log_file in ["logger.log", "web.log", "tunnel.log"]:
    if os.path.exists(log_file):
        os.remove(log_file)
        print(f"🧹 Cleaned up {log_file}")

# Configure logging with DEBUG log level.
logging.basicConfig(
    filename="logger.log",
    level=logging.DEBUG,
    format="%(filename)s:%(lineno)s %(levelname)s:%(message)s",
)

print("✅ Logging configured")

In [ ]:
!adk create observe-agent --model gemini-2.5-flash-lite --api_key $GITHUB_TOKEN

In [ ]:
%%writefile observe-agent/agent.py

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools.agent_tool import AgentTool, FunctionTool
from google.adk.tools.google_search_tool import google_search

from google.genai import types
from typing import List

retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)
compare_commits_tool = FunctionTool(func=find_missing_commits)

agent_instruction = """
You are a Git Branch Comparison Agent. Your primary function is to use the 
`find_missing_commits` tool to compare two branches in a specified GitHub 
repository. You must be provided with the repository name (e.g., 'owner/repo'), 
the base branch, and the compare branch. Your final output must clearly list the 
commits found in the compare branch that are missing from the base branch.
"""

# Create the LLM Agent
github_comp_agent = LlmAgent(
    name="github_comp_agent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction=agent_instruction,
    tools=[compare_commits_tool]
)

In [ ]:
url_prefix = get_adk_proxy_url()

In [ ]:
!adk web --log_level DEBUG --url_prefix {url_prefix}